# FAKE NEWS DETECTION – A MACHINE LEARNING BASED SYSTEM
### College Project Experimentation Notebook

**Objective:** Build, train, evaluate and visualize a Machine Learning model using **TF-IDF** and **Logistic Regression** to detect whether a news article is **REAL** or **FAKE**.

## 1. Import Required Libraries
We use **Pandas**, **NumPy**, **NLTK**, **Scikit-learn**, **Matplotlib**, and **Joblib**.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

nltk.download('stopwords', quiet=True)
print("All libraries imported successfully!")

## 2. Load and Inspect the Kaggle Dataset

In [ ]:
df = pd.read_csv('dataset/news.csv')
print("Dataset Shape:", df.shape)
df.head()

## 3. Dataset Preprocessing & Label Standardization

In [ ]:
# Combine Title and Text for complete contextual information
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
df['content'] = df['title'] + ' ' + df['text']

# Standardize Labels: 1 = FAKE, 0 = REAL
label_map = {'FAKE': 1, 'REAL': 0, '1': 1, '0': 0}
df['target'] = df['label'].astype(str).str.strip().str.upper().map(label_map)
df = df.dropna(subset=['target'])
df['target'] = df['target'].astype(int)

print("Class Counts:")
print(df['target'].value_counts())

## 4. NLP Text Cleaning & Porter Stemming (NLTK)

In [ ]:
from preprocess import preprocess_text

print("Sample raw text:")
sample = df['content'].iloc[0]
print(sample[:200])

print("\nSample cleaned text:")
print(preprocess_text(sample[:200]))

# Apply NLP preprocessing
df['cleaned_content'] = df['content'].apply(preprocess_text)

## 5. Train-Test Split (80% Train, 20% Test)

In [ ]:
X = df['cleaned_content']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training instances: {len(X_train)}")
print(f"Testing instances : {len(X_test)}")

## 6. TF-IDF Feature Extraction

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF Feature Matrix Shape: {X_train_tfidf.shape}")

## 7. Model Training: Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs', random_state=42)
model.fit(X_train_tfidf, y_train)
print("Model training finished successfully!")

## 8. Model Evaluation & Performance Metrics

In [ ]:
y_pred = model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall   : {rec * 100:.2f}%")
print(f"F1-Score : {f1 * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Real (0)', 'Fake (1)']))

## 9. Matplotlib Visualizations (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap=plt.cm.Blues)
plt.title("Confusion Matrix - Logistic Regression")
plt.colorbar()
tick_marks = np.arange(2)
plt.xticks(tick_marks, ['Real (0)', 'Fake (1)'])
plt.yticks(tick_marks, ['Real (0)', 'Fake (1)'])

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white' if cm[i, j] > cm.max()/2 else 'black', fontweight='bold')

plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

## 10. Save Model and Vectorizer with Joblib

In [ ]:
joblib.dump(model, 'model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')
print("model.pkl and vectorizer.pkl saved successfully!")